# Task 2: Technical Indicators Analysis

This notebook loads historical stock price data, computes technical indicators using TA-Lib and PyNance, and visualizes the results.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import talib
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully")

In [ ]:
# Load the stock price dataset
stock_df = pd.read_csv('../data/raw/AAPL.csv')

# Display basic information
print(f"Stock dataset shape: {stock_df.shape}")
print("\nColumn names:")
print(stock_df.columns.tolist())
print("\nFirst few rows:")
print(stock_df.head())
print("\nData types:")
print(stock_df.dtypes)

## 1. Data Preparation and Cleaning

In [ ]:
# Convert Date column to datetime
stock_df['Date'] = pd.to_datetime(stock_df['Date'])

# Set Date as index
stock_df.set_index('Date', inplace=True)

# Check for missing values
print("Missing values:")
print(stock_df.isnull().sum())

# Basic statistics
print("\nBasic statistics:")
print(stock_df.describe())

In [ ]:
# Check data quality
print(f"Date range: {stock_df.index.min()} to {stock_df.index.max()}")
print(f"Total trading days: {len(stock_df)}")

# Check for any obvious data issues
print("\nData quality checks:")
print(f"Negative prices: {(stock_df[['Open', 'High', 'Low', 'Close']] < 0).any().any()}")
print(f"High < Low: {(stock_df['High'] < stock_df['Low']).any()}")
print(f"Close outside High-Low range: {((stock_df['Close'] > stock_df['High']) | (stock_df['Close'] < stock_df['Low'])).any()}")

## 2. Technical Indicators with TA-Lib

In [ ]:
# Moving Averages
# Simple Moving Averages
stock_df['SMA_20'] = talib.SMA(stock_df['Close'].values, timeperiod=20)
stock_df['SMA_50'] = talib.SMA(stock_df['Close'].values, timeperiod=50)
stock_df['SMA_200'] = talib.SMA(stock_df['Close'].values, timeperiod=200)

# Exponential Moving Averages
stock_df['EMA_12'] = talib.EMA(stock_df['Close'].values, timeperiod=12)
stock_df['EMA_26'] = talib.EMA(stock_df['Close'].values, timeperiod=26)

print("Moving averages computed successfully")

In [ ]:
# Relative Strength Index (RSI)
stock_df['RSI'] = talib.RSI(stock_df['Close'].values, timeperiod=14)

# MACD
stock_df['MACD'], stock_df['MACD_signal'], stock_df['MACD_hist'] = talib.MACD(
    stock_df['Close'].values, fastperiod=12, slowperiod=26, signalperiod=9
)

# Bollinger Bands
stock_df['BB_upper'], stock_df['BB_middle'], stock_df['BB_lower'] = talib.BBANDS(
    stock_df['Close'].values, timeperiod=20, nbdevup=2, nbdevdn=2, matype=0
)

print("RSI, MACD, and Bollinger Bands computed successfully")

In [ ]:
# Additional technical indicators
# Average True Range (ATR)
stock_df['ATR'] = talib.ATR(
    stock_df['High'].values, stock_df['Low'].values, stock_df['Close'].values, timeperiod=14
)

# Commodity Channel Index (CCI)
stock_df['CCI'] = talib.CCI(
    stock_df['High'].values, stock_df['Low'].values, stock_df['Close'].values, timeperiod=14
)

# Stochastic Oscillator
stock_df['Stoch_k'], stock_df['Stoch_d'] = talib.STOCH(
    stock_df['High'].values, stock_df['Low'].values, stock_df['Close'].values,
    fastk_period=5, slowk_period=3, slowd_period=3
)

# Williams %R
stock_df['Williams_R'] = talib.WILLR(
    stock_df['High'].values, stock_df['Low'].values, stock_df['Close'].values, timeperiod=14
)

print("Additional technical indicators computed successfully")

## 3. Financial Metrics with PyNance

In [ ]:
# Calculate daily returns
stock_df['Daily_Return'] = stock_df['Close'].pct_change() * 100

# Calculate log returns
stock_df['Log_Return'] = np.log(stock_df['Close'] / stock_df['Close'].shift(1))

# Calculate volatility (rolling standard deviation of returns)
stock_df['Volatility_20'] = stock_df['Daily_Return'].rolling(window=20).std()
stock_df['Volatility_50'] = stock_df['Daily_Return'].rolling(window=50).std()

# Calculate price momentum
stock_df['Momentum_10'] = (stock_df['Close'] / stock_df['Close'].shift(10) - 1) * 100
stock_df['Momentum_20'] = (stock_df['Close'] / stock_df['Close'].shift(20) - 1) * 100

print("Financial metrics computed successfully")

## 4. Data Visualization

In [ ]:
# Plot closing prices with moving averages
fig, axes = plt.subplots(2, 1, figsize=(15, 12))

# Price and moving averages (last 500 days for clarity)
plot_data = stock_df.tail(500)
axes[0].plot(plot_data.index, plot_data['Close'], label='Close Price', linewidth=2)
axes[0].plot(plot_data.index, plot_data['SMA_20'], label='SMA 20', alpha=0.8)
axes[0].plot(plot_data.index, plot_data['SMA_50'], label='SMA 50', alpha=0.8)
axes[0].plot(plot_data.index, plot_data['SMA_200'], label='SMA 200', alpha=0.8)
axes[0].set_title('AAPL Stock Price with Moving Averages', fontsize=14)
axes[0].set_ylabel('Price ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(plot_data.index, plot_data['Volume'], alpha=0.6, color='gray')
axes[1].set_title('Trading Volume', fontsize=14)
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot RSI and MACD
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# RSI
axes[0].plot(plot_data.index, plot_data['RSI'], label='RSI', color='purple')
axes[0].axhline(y=70, color='red', linestyle='--', alpha=0.7, label='Overbought (70)')
axes[0].axhline(y=30, color='green', linestyle='--', alpha=0.7, label='Oversold (30)')
axes[0].set_title('Relative Strength Index (RSI)', fontsize=14)
axes[0].set_ylabel('RSI')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MACD
axes[1].plot(plot_data.index, plot_data['MACD'], label='MACD', color='blue')
axes[1].plot(plot_data.index, plot_data['MACD_signal'], label='Signal Line', color='red')
axes[1].bar(plot_data.index, plot_data['MACD_hist'], label='Histogram', alpha=0.6, color='gray')
axes[1].set_title('MACD (Moving Average Convergence Divergence)', fontsize=14)
axes[1].set_ylabel('MACD')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Bollinger Bands
axes[2].plot(plot_data.index, plot_data['Close'], label='Close Price', color='black', linewidth=1)
axes[2].plot(plot_data.index, plot_data['BB_upper'], label='Upper Band', color='red', alpha=0.7)
axes[2].plot(plot_data.index, plot_data['BB_middle'], label='Middle Band', color='blue', alpha=0.7)
axes[2].plot(plot_data.index, plot_data['BB_lower'], label='Lower Band', color='green', alpha=0.7)
axes[2].fill_between(plot_data.index, plot_data['BB_upper'], plot_data['BB_lower'], alpha=0.1, color='gray')
axes[2].set_title('Bollinger Bands', fontsize=14)
axes[2].set_ylabel('Price ($)')
axes[2].set_xlabel('Date')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot additional indicators
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Daily Returns
axes[0,0].plot(plot_data.index, plot_data['Daily_Return'], color='blue', alpha=0.7)
axes[0,0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0,0].set_title('Daily Returns (%)', fontsize=12)
axes[0,0].set_ylabel('Return (%)')
axes[0,0].grid(True, alpha=0.3)

# Volatility
axes[0,1].plot(plot_data.index, plot_data['Volatility_20'], label='20-day Volatility', color='orange')
axes[0,1].plot(plot_data.index, plot_data['Volatility_50'], label='50-day Volatility', color='purple')
axes[0,1].set_title('Rolling Volatility', fontsize=12)
axes[0,1].set_ylabel('Volatility')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Stochastic Oscillator
axes[1,0].plot(plot_data.index, plot_data['Stoch_k'], label='%K', color='blue')
axes[1,0].plot(plot_data.index, plot_data['Stoch_d'], label='%D', color='red')
axes[1,0].axhline(y=80, color='red', linestyle='--', alpha=0.5, label='Overbought')
axes[1,0].axhline(y=20, color='green', linestyle='--', alpha=0.5, label='Oversold')
axes[1,0].set_title('Stochastic Oscillator', fontsize=12)
axes[1,0].set_ylabel('Value')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Williams %R
axes[1,1].plot(plot_data.index, plot_data['Williams_R'], color='green')
axes[1,1].axhline(y=-20, color='red', linestyle='--', alpha=0.5, label='Overbought')
axes[1,1].axhline(y=-80, color='green', linestyle='--', alpha=0.5, label='Oversold')
axes[1,1].set_title('Williams %R', fontsize=12)
axes[1,1].set_ylabel('Value')
axes[1,1].set_xlabel('Date')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Indicator Analysis and Insights

In [ ]:
# Generate trading signals based on indicators
def generate_signals(df):
    signals = pd.DataFrame(index=df.index)
    
    # Moving Average Crossover
    signals['MA_Cross'] = 0
    signals.loc[df['SMA_20'] > df['SMA_50'], 'MA_Cross'] = 1  # Bullish
    signals.loc[df['SMA_20'] < df['SMA_50'], 'MA_Cross'] = -1  # Bearish
    
    # RSI signals
    signals['RSI_Signal'] = 0
    signals.loc[df['RSI'] > 70, 'RSI_Signal'] = -1  # Overbought - sell signal
    signals.loc[df['RSI'] < 30, 'RSI_Signal'] = 1   # Oversold - buy signal
    
    # MACD signals
    signals['MACD_Signal'] = 0
    signals.loc[df['MACD'] > df['MACD_signal'], 'MACD_Signal'] = 1  # Bullish
    signals.loc[df['MACD'] < df['MACD_signal'], 'MACD_Signal'] = -1  # Bearish
    
    return signals

signals_df = generate_signals(stock_df)

# Show recent signals
print("Recent trading signals:")
print(signals_df.tail(10))

In [ ]:
# Summary statistics for indicators
print("Technical Indicators Summary Statistics:")
indicators = ['RSI', 'MACD', 'MACD_signal', 'ATR', 'CCI', 'Stoch_k', 'Stoch_d', 'Williams_R']
print(stock_df[indicators].describe())

In [ ]:
# Correlation analysis between indicators
correlation_matrix = stock_df[indicators + ['Daily_Return']].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f', cbar_kws={'label': 'Correlation'})
plt.title('Correlation Matrix of Technical Indicators', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Key Insights Summary

In [ ]:
# Generate key insights
insights = {
    'Current Price': f"${stock_df['Close'].iloc[-1]:.2f}",
    '52-Week High': f"${stock_df['Close'].rolling(252).max().iloc[-1]:.2f}",
    '52-Week Low': f"${stock_df['Close'].rolling(252).min().iloc[-1]:.2f}",
    'Current RSI': f"{stock_df['RSI'].iloc[-1]:.2f}",
    'RSI Status': 'Overbought' if stock_df['RSI'].iloc[-1] > 70 else 'Oversold' if stock_df['RSI'].iloc[-1] < 30 else 'Neutral',
    'Current MACD': f"{stock_df['MACD'].iloc[-1]:.4f}",
    'MACD Signal': 'Bullish' if stock_df['MACD'].iloc[-1] > stock_df['MACD_signal'].iloc[-1] else 'Bearish',
    '20-day Volatility': f"{stock_df['Volatility_20'].iloc[-1]:.4f}",
    'MA Trend': 'Bullish' if stock_df['SMA_20'].iloc[-1] > stock_df['SMA_50'].iloc[-1] else 'Bearish'
}

print("Technical Analysis Summary:")
for key, value in insights.items():
    print(f"{key}: {value}")

### Key Findings:

1. **Price Trends**: The stock is currently trading [above/below] key moving averages, indicating [bullish/bearish] momentum
2. **RSI Analysis**: Current RSI of [value] suggests the stock is [overbought/oversold/neutral]
3. **MACD Signals**: MACD line is [above/below] signal line, indicating [bullish/bearish] momentum
4. **Volatility**: Current 20-day volatility of [value] suggests [high/low] market uncertainty
5. **Support/Resistance**: Bollinger Bands show current price trading [within/outside] normal range

This technical analysis provides a foundation for correlation with news sentiment in Task 3.